# Phase 10 - generate Text-to-SQL predictions with the fine-tuned adapter

Runs **unattended**. `Save Version` -> `Save & Run All (Commit)`, close the
tab, come back to `predictions.jsonl`.

## Set these in the right-hand panel first

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Internet | **On** (the 16 GB base model is downloaded) |
| Input 1 | the **output of the training notebook** (holds `final_adapter/`) |
| Input 2 | the **`text2sql-evalpack`** dataset |

## What this notebook does *not* do

It does not score anything. It has no gold SQL, no database, and no metrics
code - only the 453 questions. Scoring happens on the laptop, against
PostgreSQL, through the same harness that produced the frozen 10.82 %
baseline. Splitting it this way is what makes the comparison honest: the
model cannot be tuned against a score it never sees.


## 1. Environment and inputs

Fails immediately if the GPU or either input is missing, rather than after a
16 GB download.


In [ ]:
import os, sys, json, glob

import torch
assert torch.cuda.is_available(), 'No GPU - set Accelerator to GPU T4 x2'
p = torch.cuda.get_device_properties(0)
print(f'{p.name} | {p.total_memory/1024**3:.1f} GB | compute {p.major}.{p.minor}')

print()
print('--- /kaggle/input ---')
for root, dirs, files in os.walk('/kaggle/input'):
    for f in sorted(files):
        print(' ', os.path.join(root, f))

def find(name):
    hits = [os.path.join(r, name)
            for r, _, fs in os.walk('/kaggle/input') if name in fs]
    if not hits:
        raise FileNotFoundError(
            f'{name} not found under /kaggle/input - check the Input panel')
    return hits[0]

ADAPTER_DIR = os.path.dirname(find('adapter_config.json'))
QUESTIONS_F = find('eval_questions.jsonl')
SCHEMA_F    = find('schema_context.txt')
PROMPT_F    = find('prompt_module.py')
MANIFEST_F  = find('manifest.json')

print()
print('adapter  :', ADAPTER_DIR)
print('evalpack :', os.path.dirname(QUESTIONS_F))


## 2. Install dependencies

Loose lower bounds on purpose - pinning exact versions is what broke the
training run on Colab. `peft` is needed here to apply the adapter.


In [ ]:
!pip install -q -U bitsandbytes transformers peft accelerate 2>&1 | tail -3

import importlib.metadata as _md
for pkg in ['torch','transformers','peft','bitsandbytes','accelerate']:
    try:    print(f'{pkg:<15}{_md.version(pkg)}')
    except Exception: print(f'{pkg:<15}MISSING')


## 3. Verify the eval pack

The prompt and the schema must be byte-identical to what the frozen
baseline used. If either differs, the run measures prompt engineering or a
changed database rather than fine-tuning, so this aborts rather than warns.

The fingerprints are recomputed here from the shipped files - not merely
read out of the manifest - so a corrupted or edited file is caught.


In [ ]:
import hashlib, importlib.util

EXPECTED_PROMPT_FP = '8288e41a496531a9'
EXPECTED_SCHEMA_FP = 'd03619e711661bc5'

spec = importlib.util.spec_from_file_location('prompt_module', PROMPT_F)
prompt_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prompt_module)
build_messages = prompt_module.build_messages

SCHEMA = open(SCHEMA_F, encoding='utf-8').read()
schema_fp = hashlib.sha256(SCHEMA.encode('utf-8')).hexdigest()[:16]
prompt_fp = prompt_module.prompt_fingerprint()

QUESTIONS = [json.loads(l) for l in open(QUESTIONS_F, encoding='utf-8') if l.strip()]
MANIFEST = json.load(open(MANIFEST_F, encoding='utf-8'))

# Configuration 3 shows every question the whole schema; configuration 4
# shows each question only the tables a retriever picked. The pack says
# which, so one notebook serves both and they cannot be confused.
SCHEMA_MODE = MANIFEST['schema'].get('mode', 'full')
PER_QUESTION_SCHEMA = SCHEMA_MODE == 'retrieved'


def schema_for(q):
    return q['schema'] if PER_QUESTION_SCHEMA else SCHEMA


print(f'prompt      {prompt_module.PROMPT_VERSION}  {prompt_fp}  '
      f"{'MATCH' if prompt_fp == EXPECTED_PROMPT_FP else 'MISMATCH'}")
print(f'schema mode {SCHEMA_MODE}')
print(f'questions   {len(QUESTIONS):,}')

assert prompt_fp == EXPECTED_PROMPT_FP, 'prompt differs from the frozen baseline'
assert len(QUESTIONS) == MANIFEST['questions']['count']

if PER_QUESTION_SCHEMA:
    combined = hashlib.sha256()
    for q in QUESTIONS:
        combined.update(q['id'].encode()); combined.update(bytes([0]))
        combined.update(q['schema'].encode()); combined.update(bytes([1]))
    got = combined.hexdigest()[:16]
    want = MANIFEST['schema']['fingerprint']
    sizes = [len(q['schema']) for q in QUESTIONS]
    print(f'retrieved   {got}  '
          f"{'MATCH' if got == want else 'MISMATCH'}"
          f'  mean {sum(sizes)/len(sizes):,.0f} chars '
          f'(full schema is {len(SCHEMA):,})')
    assert got == want, 'retrieved schemas differ from the export'
    allowed = {'id', 'question', 'schema'}
else:
    print(f'schema      {len(SCHEMA):,} chars  {schema_fp}  '
          f"{'MATCH' if schema_fp == EXPECTED_SCHEMA_FP else 'MISMATCH'}")
    assert schema_fp == EXPECTED_SCHEMA_FP, 'schema differs from the frozen baseline'
    allowed = {'id', 'question'}

# Structural proof that no answers travelled with the questions.
extra = {k for q in QUESTIONS for k in q} - allowed
assert not extra, f'eval pack carries unexpected fields: {extra}'
assert MANIFEST['questions']['gold_sql_included'] is False
print()
print('verified - questions only, no gold SQL present')
print('baseline to beat:', MANIFEST['baseline_to_beat'])


## 4. Load the base model in 4-bit and apply the adapter

Same quantisation as training: NF4 with double quantisation, fp16 compute
(a T4 is Turing and has no bf16). The tokenizer comes from the adapter
directory so the chat template is exactly the one training used.

The 4-bit load is asserted rather than assumed. A silently-dropped
`quantization_config` is what caused the first training attempts to load in
full precision and get killed.


In [ ]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_ID  = MANIFEST.get('base_model', 'Qwen/Qwen3-8B')
REVISION = 'b968826d9c46'   # pinned: same weights the adapter was trained on

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)

# The adapter's tokenizer_config.json has no inline chat_template - the
# template lives in chat_template.jinja beside it. Recent transformers
# picks that up automatically, older versions do not, and a tokenizer with
# no template silently falls back to a default that the model never saw.
if not getattr(tok, 'chat_template', None):
    tpl_path = os.path.join(ADAPTER_DIR, 'chat_template.jinja')
    tok.chat_template = open(tpl_path, encoding='utf-8').read()
    print('chat template loaded explicitly from chat_template.jinja')
assert tok.chat_template, 'no chat template - rendering cannot match training'
assert 'enable_thinking' in tok.chat_template, (
    'this template has no enable_thinking branch - it is not the Qwen3 template the adapter was trained with')

if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = 'left'   # required for batched decoder-only generation

t0 = time.perf_counter()
base = AutoModelForCausalLM.from_pretrained(
    BASE_ID, revision=REVISION, quantization_config=bnb,
    device_map={'': 0}, torch_dtype=torch.float16,
    attn_implementation='sdpa',
)

n4 = sum(1 for m in base.modules() if m.__class__.__name__ == 'Linear4bit')
assert n4 > 0, 'model did NOT load in 4-bit - quantization_config was dropped'
print(f'4-bit load confirmed | {n4} Linear4bit modules | '
      f'VRAM {torch.cuda.memory_allocated()/1024**3:.2f} GB')

model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
n_lora = sum(1 for n, _ in model.named_parameters() if 'lora_' in n)
assert n_lora > 0, 'adapter did not attach - no lora_ parameters found'
print(f'adapter attached  | {n_lora} LoRA tensors | '
      f'VRAM {torch.cuda.memory_allocated()/1024**3:.2f} GB | '
      f'{time.perf_counter()-t0:.0f}s')


## 5. Render the prompt exactly as training did

Qwen3's chat template renders a *finished* assistant turn as

```
<|im_start|>assistant
<think>

</think>

SELECT ...
```

so every training target was preceded by an empty think block. With
`add_generation_prompt=True` and no `enable_thinking` argument the template
stops at `<|im_start|>assistant
` and leaves the model to produce
`<think>...` itself - a starting point it never saw during training. That
produced reasoning prose, a stray `</think>`, and junk tokens ahead of the
SQL on the first attempt at this notebook.

`enable_thinking=False` appends the empty block, so generation resumes at
exactly the token the SQL started on in training.

The assertion below is deliberately strict. Checking only that the inference
prompt is a *prefix* of the training rendering is not enough - it was, and
the run was still wrong. What matters is that nothing sits in the gap
between them.


In [ ]:
SENTINEL = 'SELECT 1'


def render_prompt(question, schema):
    """The single definition of how a prompt is rendered."""
    return tok.apply_chat_template(
        build_messages(question, schema), tokenize=False,
        add_generation_prompt=True, enable_thinking=False)


demo = build_messages(QUESTIONS[0]['question'], schema_for(QUESTIONS[0]))
infer_text = render_prompt(QUESTIONS[0]['question'], schema_for(QUESTIONS[0]))
train_text = tok.apply_chat_template(
    demo + [{'role': 'assistant', 'content': SENTINEL}], tokenize=False)

is_prefix = train_text.startswith(infer_text)
gap = train_text[len(infer_text):] if is_prefix else None
resumes_at_sql = bool(gap) and gap.startswith(SENTINEL)

print('inference prompt is a prefix of the training rendering:', is_prefix)
print('generation resumes exactly at the SQL              :', resumes_at_sql)
if is_prefix and not resumes_at_sql:
    filler = gap.split(SENTINEL)[0]
    print()
    print('!! the model would have to emit this first:', repr(filler))

assert is_prefix, 'prompt rendering diverges from the training format'
assert resumes_at_sql, (
    'the model would have to generate filler before the SQL - it was never '
    'trained to do that, and the output would not be scoreable')

n_tok = len(tok(infer_text, add_special_tokens=False)['input_ids'])
print(f'rendered prompt: {len(infer_text):,} chars, {n_tok:,} tokens')
print()
print('--- last 200 chars of the prompt the model will see ---')
print(repr(infer_text[-200:]))


## 6. Generate

Greedy decoding (`do_sample=False`), `max_new_tokens=512` - the same
settings as the frozen baseline's `temperature=0, max_tokens=512`. Sampling
here would make the comparison partly a coin toss.

Batched with left padding, checkpointed after every batch. A session cut
short loses at most one batch: re-running this cell resumes from the file.
On an out-of-memory error the batch size halves and retries rather than
dying.


In [ ]:
# Named after the configuration so a retrieved-schema run cannot be
# mistaken for a full-schema one after download.
PRED = ('/kaggle/working/predictions.jsonl' if not PER_QUESTION_SCHEMA
        else '/kaggle/working/predictions_retrieved.jsonl')
BATCH = 8
MAX_NEW = 512
PROMPT_FORMAT = 'enable_thinking=False'   # stamped into the header so a
                                          # resume cannot mix formats

header = {'_header': True,
          'prompt_fingerprint': prompt_fp,
          'schema_fingerprint': schema_fp,
          'base_model': BASE_ID, 'revision': REVISION,
          'adapter_dir': ADAPTER_DIR,
          'decoding': 'greedy', 'max_new_tokens': MAX_NEW,
          'prompt_format': PROMPT_FORMAT,
          'schema_mode': SCHEMA_MODE,
          'gpu': p.name}

# Resume must never splice together rows generated under different prompt
# formats. The first version of this notebook omitted enable_thinking=False
# and produced unusable output; silently topping that file up would hide
# half a broken run inside a good one.
done = {}
if os.path.exists(PRED):
    lines = [json.loads(l) for l in open(PRED, encoding='utf-8') if l.strip()]
    old_header = next((r for r in lines if r.get('_header')), {})
    if old_header.get('prompt_format') != PROMPT_FORMAT:
        stale = PRED + '.stale'
        os.replace(PRED, stale)
        print('existing file used prompt_format',
              repr(old_header.get('prompt_format')),
              '- moved aside, starting fresh')
    else:
        done = {r['example_id']: r for r in lines if not r.get('_header')}
        print(f'resuming: {len(done)} already generated')
if not os.path.exists(PRED):
    with open(PRED, 'w', encoding='utf-8') as fh:
        fh.write(json.dumps(header) + chr(10))

todo = [q for q in QUESTIONS if q['id'] not in done]
print(f'to generate: {len(todo)} of {len(QUESTIONS)}')


def run_batch(chunk):
    texts = [render_prompt(q['question'], schema_for(q)) for q in chunk]
    enc = tok(texts, return_tensors='pt', padding=True,
              add_special_tokens=False).to(model.device)
    n_in = enc['input_ids'].shape[1]
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tok.pad_token_id)
    per_ms = (time.perf_counter() - t0) * 1000 / len(chunk)
    rows = []
    for j, q in enumerate(chunk):
        gen_ids = out[j][n_in:]
        keep = [t for t in gen_ids.tolist() if t != tok.pad_token_id]
        rows.append({
            'example_id': q['id'],
            'raw_output': tok.decode(gen_ids, skip_special_tokens=True),
            'latency_ms': round(per_ms, 2),
            'ok': True, 'error': None,
            'finish_reason': 'length' if len(keep) >= MAX_NEW else 'stop',
            'prompt_tokens': int(enc['attention_mask'][j].sum()),
            'completion_tokens': len(keep),
        })
    return rows


started = time.perf_counter()
fh = open(PRED, 'a', encoding='utf-8')
i = 0
size = BATCH   # sticky: once VRAM says 8 is too wide, it stays too wide,
               # so a reset would burn a failed prefill on every batch
while i < len(todo):
    while True:
        chunk = todo[i:i + size]
        try:
            rows = run_batch(chunk)
            break
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache()
            if size == 1:
                rows = [{'example_id': q['id'], 'raw_output': '',
                         'latency_ms': 0.0, 'ok': False,
                         'error': 'CUDA OOM at batch size 1',
                         'finish_reason': None, 'prompt_tokens': None,
                         'completion_tokens': 0} for q in chunk]
                break
            size = max(1, size // 2)
            print(f'  OOM -> retrying at batch size {size}', flush=True)
    for r in rows:
        fh.write(json.dumps(r, ensure_ascii=False) + chr(10))
    fh.flush()
    i += len(rows)
    if (i // BATCH) % 5 == 0 or i >= len(todo):
        el = time.perf_counter() - started
        eta = el / max(i, 1) * (len(todo) - i)
        print(f'  {i:>4}/{len(todo)}  elapsed {el/60:5.1f}m  '
              f'eta {eta/60:5.1f}m', flush=True)
fh.close()
print()
print(f'done in {(time.perf_counter()-started)/60:.1f} minutes')


## 7. Sanity-check the output, then package it

A quick look before downloading: does the output actually contain SQL, and
did anything hit the token limit? Nothing here is scored - `SELECT`
appearing in the text says only that the model produced *a* query, not a
*correct* one. That verdict comes from executing it against PostgreSQL on
the laptop.


In [ ]:
rows = [json.loads(l) for l in open(PRED, encoding='utf-8') if l.strip()]
preds = [r for r in rows if not r.get('_header')]

looks_like_sql = sum(1 for r in preds
                     if 'SELECT' in r['raw_output'].upper())
truncated = sum(1 for r in preds if r['finish_reason'] == 'length')
failed = sum(1 for r in preds if not r['ok'])
avg_out = sum(r['completion_tokens'] for r in preds) / max(len(preds), 1)
avg_ms = sum(r['latency_ms'] for r in preds) / max(len(preds), 1)

print(f'predictions        {len(preds):>6} of {len(QUESTIONS)}')
print(f'contain SELECT     {looks_like_sql:>6}')
print(f'hit token limit    {truncated:>6}')
print(f'generation failed  {failed:>6}')
print(f'mean out tokens    {avg_out:>9.1f}')
print(f'mean latency       {avg_ms:>9.0f} ms')
print()
for r in preds[:3]:
    print('-' * 68)
    print(r['example_id'], '|', repr(r['raw_output'][:200]))

import shutil
shutil.copyfile(PRED, '/kaggle/working/predictions_final.jsonl')
print()
print('-' * 68)
print('download predictions.jsonl from the Output tab')
print(f"size: {os.path.getsize(PRED)/1e6:.2f} MB")


---

## Next step, on the laptop

```powershell
env\Scripts\python.exe scripts/score_finetuned.py `
    --predictions "C:\Users\dell\Downloads\predictions.jsonl"
```

That applies the baseline's own `extract_sql`, executes every query against
PostgreSQL in a read-only session, compares result fingerprints, and prints
the head-to-head against 10.82 %.

## If the session is cut short

`predictions.jsonl` persists in the notebook output. Re-run cell 6 and it
resumes from where it stopped - at most one batch is lost.
